# Stage 6: 多方法注释 + 跨方法比较

对每个 Leiden 簇并行运行多种标注方法，交叉比对结果。
**多方法并跑不是冗余——不同独立方法的共识最能提高置信度**，
分歧标记需要 PI 重点复核的簇。

本 notebook 产出：
- 各方法独立标注列（`cell_type_{method}_v1`）
- 成对混淆矩阵 + Cohen's kappa
- 每簇 LLM 综合判决 markdown
- PI 最终标注列 `cell_type_final_v1`

# 注：SPEC 早期提到的 sweep recommendations cell 依赖已删除的 sweep 函数（见 ADR-0009 去封装），本 notebook 不含；如需参数扫描建议，PI 可在 jupyter 内手写显式循环。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：Stage 5（多分辨率 Leiden 聚类），读 `stage5_clustered_v*.h5ad`
- **下游**：Stage 6.5（亚群 Subset 重分析）/ Stage 7（下游分析），产出 `stage6_annotated_v*.h5ad`

### 为什么要迭代回跑？
注释质量直接影响下游逐簇分析和亚群重分析的准确性。如果在 stage 6.5（亚群分析发现注释不合理）、
stage 7（跨病种比较时发现标签粒度不对）或逐簇报告中发现问题，可能需要：
- 换用不同的 Leiden 分辨率的列（修改 `LEIDEN_COL`）
- 增加或替换标记物 CSV（修改 `MARKER_CSV`）
- 调整 LLM 模型选择（修改 `LLM_MODELS` / `CONSENSUS_MODEL`）
- 在 PI 手动标注区修改 `marker_assignments` 或 `pi_decisions` 的标签
- 换用 stage 5 的另一个版本（不同 Leiden 分辨率组合）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `nancang_stage5_clustered_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `nancang_stage6_annotated_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `LEIDEN_COL`、`MARKER_CSV`、
   或更新 `marker_assignments` / `pi_decisions`）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为注释质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：stage 6.5 和 stage 7 的 `UPSTREAM_PATH` 指向你决定采用的 stage 6 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"stage6_annotated"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"stage 6 有哪些版本？哪些依赖 stage5_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH   -- stage 5（已聚类）输出
# OUTPUT_PATH     -- 本 stage 产出 checkpoint。
#                        版本号 _v1 与 adata.uns["version"] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# MARKER_CSV      -- 标记物知识库 CSV（供 dotplot + 基因集评分）
# LEIDEN_COL      -- 用作簇标签的 obs 列
# LLM_MODELS      -- mLLMCelltype 用的模型列表（按模型名前缀路由）
# CONSENSUS_MODEL -- mLLMCelltype 共识讨论模型
# REFERENCE_ATLAS_PATH -- scANVI 标注参考 atlas（不存在则跳过）
# BASE_URLS       -- 各家 provider 的 base URL 字典

UPSTREAM_PATH = "results/nancang_stage5_clustered_v1.h5ad"
OUTPUT_PATH   = "results/nancang_stage6_annotated_v1.h5ad"

MARKER_CSV  = "references/markers/gastric_TEST_markers.csv"  # 测试夹具：PI 将在 PR-5 替换为真实 marker 库
LEIDEN_COL  = "leiden_res_0.6"

# mLLMCelltype 多模型共识（写代码不测试，PI 配 .env key 后人工运行调试）
LLM_MODELS = [
    "openai/gpt-4.1-nano",      # OpenAI 前缀 → 走 OPENAI_API_KEY
    "anthropic/claude-haiku-4-5",# Anthropic 前缀 → 走 ANTHROPIC_API_KEY
    "deepseek/deepseek-chat",    # DeepSeek 前缀 → 走 DEEPSEEK_API_KEY
]
CONSENSUS_MODEL = "anthropic/claude-sonnet-4-6"  # 共识讨论用更强模型

# scANVI 参考 atlas（现在没有胃粘膜标注参考，跳过）
REFERENCE_ATLAS_PATH = ""  # 留空 = 跳过；填路径 = 启用标签迁移

# 各家 provider 的 base URL（mLLMCelltype 按模型名前缀自动路由）
# 留空字符串 = 用官方默认；国内用户可能需要代理 URL
BASE_URLS = {
    "openai":     "",   # 官方 https://api.openai.com/v1
    "anthropic":  "",
    "deepseek":   "https://api.deepseek.com",
    "qwen":       "",   # 国内：https://dashscope.aliyuncs.com/compatible-mode/v1
    "gemini":     "",
}

# LLM verdict per cluster 用的模型（同 key 守卫，待 PI 配 key）
VERDICT_MODEL = "anthropic/claude-sonnet-4-6"

In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
import sys, os, gc
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures/stage6_verdicts", exist_ok=True)
os.makedirs("results/figures/stage6_sankey", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

In [ ]:
# 导入（scanpy 原生 API + 框架函数）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import datetime, warnings

# 框架函数
from scrna_integration import load_markers
from scrna_integration.scorers import annotation_concordance

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"obsm 键: {list(adata.obsm.keys())}")
print(f"leiden 列 '{LEIDEN_COL}' 存在: {LEIDEN_COL in adata.obs.columns}")
if LEIDEN_COL in adata.obs.columns:
    print(f"  簇数: {adata.obs[LEIDEN_COL].nunique()}")

In [ ]:
# 加载标记物知识库（marker CSV 不存在时优雅跳过，待 PR-5 正式建库）。
if os.path.exists(MARKER_CSV):
    markers = load_markers(MARKER_CSV)
    print(f"标记物库: {MARKER_CSV}")
    print(f"  细胞类型数: {len(markers)}")
    for ct, genes in list(markers.items())[:5]:
        print(f"  {ct}: {genes}")
    if len(markers) > 5:
        print(f"  ... 共 {len(markers)} 种细胞类型")

    # 展平为所有标记基因列表（用于 dotplot）
    all_marker_genes = sorted(set(g for glist in markers.values() for g in glist))
    # 只保留在 adata 中实际存在的基因
    available_markers = [g for g in all_marker_genes if g in adata.var_names]
    missing = set(all_marker_genes) - set(available_markers)
    if missing:
        print(f"  数据中不存在的标记基因（跳过）: {sorted(missing)}")
    print(f"  可用标记基因: {len(available_markers)}/{len(all_marker_genes)}")
else:
    print(f"marker CSV 不存在 ({MARKER_CSV})，跳过 marker 注释；待 PR-5 正式建库")
    markers = {}
    available_markers = []


## 方法 1：标记物 dotplot（PI 手动标注）

最直观的方法——用已有标记物知识库画 dotplot，PI 看每个簇表达哪些标记物，
然后手动填入 `marker_assignments` 字典。
**为什么先做这个？** 让 PI 在受自动方法影响前建立自己的判断，避免锚定偏差。
（也可以后做——方法顺序不影响结果，PI 自由选择。）

In [ ]:
# 标记物 dotplot：每个簇 × 每个标记基因的表达百分比 + 平均表达量。
# PI 浏览此图后填写 marker_assignments 字典。
if available_markers and LEIDEN_COL in adata.obs.columns:
    sc.pl.dotplot(
        adata, var_names=available_markers, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot ({LEIDEN_COL})",
        save="_stage6_dotplot.png",
    )
    # 移动 scanpy 默认 figures/ 到 results/figures/
    # scanpy save 行为：前缀 dotplot_ + save 值 = dotplot__stage6_dotplot.png
    src = "figures/dotplot__stage6_dotplot.png"
    dst = "results/figures/stage6_dotplot.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"已保存 dotplot: {dst}")
    plt.close("all")
else:
    print("无可用的标记基因或缺少 leiden 列，跳过 dotplot")

In [ ]:
# === PI 手动标注区 ===
# PI 查看上方 dotplot 后，在下方字典中为每个簇 ID 填入细胞类型。
# 示例（基于 Nowicki 数据——PI 需根据实际 dotplot 修改）：
marker_assignments = {
    # "0": "B_cell",
    # "1": "T_cell",
    # "2": "myeloid",
    # ...  PI 逐簇填入
}

if marker_assignments:
    adata.obs["cell_type_marker_v1"] = (
        adata.obs[LEIDEN_COL].astype(str).map(marker_assignments)
    )
    n_assigned = adata.obs["cell_type_marker_v1"].notna().sum()
    print(f"marker 标注: {n_assigned}/{adata.n_obs} 细胞已标注")
else:
    # PI 暂未填写，预建空列保持结构完整
    adata.obs["cell_type_marker_v1"] = np.nan
    adata.obs["cell_type_marker_v1"] = (
        adata.obs["cell_type_marker_v1"].astype("category")
    )
    print("marker 标注: PI 暂未填写，已预建 cell_type_marker_v1 空列")

## 方法 2：mLLMCelltype 多模型共识

每簇取 top 标记基因 → 多个 LLM 独立判断细胞类型 → 共识讨论 → 产出一致注释。
**为什么多模型共识？** 单模型可能被训练数据偏差误导；多模型独立给出判断后
再由 consensus model 主持讨论，分歧的簇会被标记为 `controversial_clusters`，
PI 重点复核。

**状态：代码已写，待 PI 在 `.env` 配 key 后人工运行调试。**
无 key 时自动跳过，notebook 不崩溃。

In [ ]:
# mLLMCelltype 多模型共识注释（key 守卫——无 key 跳过）。
# 用法：mct.interactive_consensus_annotation(
#     marker_genes={cluster_id: [top_genes]}, species="human", tissue="stomach",
#     models=LLM_MODELS, consensus_model=CONSENSUS_MODEL,
#     consensus_threshold=0.7, entropy_threshold=1.0, max_discussion_rounds=3,
#     base_urls=BASE_URLS)
# 产出：result["consensus"] / ["consensus_proportion"] / ["entropy"] /
#        ["controversial_clusters"] / ["discussion_logs"]

from dotenv import load_dotenv
load_dotenv()

# 检查是否有至少一家 LLM key 已配置
_llm_providers = ["OPENAI", "ANTHROPIC", "DEEPSEEK", "QWEN", "GEMINI"]
_has_key = any(
    os.getenv(f"{p}_API_KEY") not in (None, "")
    for p in _llm_providers
)

if _has_key:
    import mllmcelltype as mct

    # 每簇取 top 标记基因（用 scanpy 原生 rank_genes_groups）
    # 为什么 n_genes=30？足够覆盖主要特征又不引入过多噪声基因
    sc.tl.rank_genes_groups(
        adata, groupby=LEIDEN_COL, method="wilcoxon",
        n_genes=30, key_added="rank_genes_stage6",
    )

    # 构建 mLLMCelltype 所需的 marker_genes 字典
    # 兜底：reload 后 leiden 列可能非 categorical
    if not hasattr(adata.obs[LEIDEN_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LEIDEN_COL]):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype("category")
    _cluster_ids = adata.obs[LEIDEN_COL].cat.categories.tolist()
    _marker_dict = {}
    for _cid in _cluster_ids:
        _df = sc.get.rank_genes_groups_df(
            adata, group=_cid, key="rank_genes_stage6"
        )
        _marker_dict[_cid] = _df["names"].head(20).tolist()
    print(f"为 {len(_marker_dict)} 个簇提取了 top 标记基因")

    # 调用 mLLMCelltype 多模型共识
    print("启动 mLLMCelltype 共识注释（多模型讨论可能需要数分钟）...")
    _result = mct.interactive_consensus_annotation(
        marker_genes=_marker_dict,
        species="human",
        tissue="stomach",
        models=LLM_MODELS,
        consensus_model=CONSENSUS_MODEL,
        consensus_threshold=0.7,
        entropy_threshold=1.0,
        max_discussion_rounds=3,
        base_urls=BASE_URLS,
    )

    # 写回 obs 列
    _consensus_map = _result.get("consensus", {})
    adata.obs["cell_type_llm_v1"] = (
        adata.obs[LEIDEN_COL].astype(str).map(_consensus_map)
    )

    # 记录共识比例（confidence）
    _prop_map = _result.get("consensus_proportion", {})
    _ent_map = _result.get("entropy", {})
    adata.obs["cell_type_llm_v1_proportion"] = (
        adata.obs[LEIDEN_COL].astype(str).map(_prop_map)
    )
    adata.obs["cell_type_llm_v1_entropy"] = (
        adata.obs[LEIDEN_COL].astype(str).map(_ent_map)
    )

    # 记录有争议的簇
    _controversial = _result.get("controversial_clusters", [])
    print(f"LLM 共识: {len(_consensus_map)} 个簇已标注")
    print(f"  有争议簇（需 PI 重点复核）: {_controversial}")
    if _prop_map:
        print(f"  共识比例范围: {min(_prop_map.values())} - {max(_prop_map.values())}")
    adata.uns["cell_type_llm_v1_meta"] = {
        "method": "mLLMCelltype",
        "models": LLM_MODELS,
        "consensus_model": CONSENSUS_MODEL,
        "controversial_clusters": _controversial,
        "timestamp": datetime.datetime.now().isoformat(),
    }
else:
    # 无 key —— 优雅跳过，不崩溃
    print("=" * 60)
    print("mLLMCelltype 共识注释已跳过——未检测到 LLM API key。")
    print("请在项目根目录 .env 文件中配置至少一家 provider 的 API key，")
    print("然后重新运行本 cell。模板见 .env.example。")
    print("配置后在 notebook 中重新执行本 cell 即可。")
    print("=" * 60)
    # 预建空列保持结构完整
    adata.obs["cell_type_llm_v1"] = np.nan
    adata.obs["cell_type_llm_v1"] = (
        adata.obs["cell_type_llm_v1"].astype("category")
    )

## 方法 3：基因集评分

用 `sc.tl.score_genes` 对每类标记基因集合做评分，
得到每个细胞相对于每个细胞类型的连续得分（`obs["score_{celltype}"]`）。
**为什么做基因集评分？** 评分提供了连续性证据——一个簇可能同时高表达多种
细胞类型的标记，说明该簇可能是过渡态或混合群体。这作为交叉比对的补充证据，
不作为独立的细胞类型标签。

In [ ]:
# 对每个细胞类型做基因集评分（sc.tl.score_genes）。
# score_genes 对每个细胞计算：标记基因平均表达 - 随机参考基因平均表达。
# 为什么用 score_genes 而非 AUCell？score_genes 是 scanpy 原生，
# 零额外依赖，学生直接看懂。AUCell 速度更慢且需要额外安装。

_score_cols = []
for ct, gene_list in markers.items():
    # 只保留数据中实际存在的基因
    _present = [g for g in gene_list if g in adata.var_names]
    if len(_present) < 2:
        print(f"  {ct}: 可用标记基因 <2（{len(_present)}），跳过评分")
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
    print(f"  {ct}: {len(_present)}/{len(gene_list)} 个基因可用 -> obs['{col}']")

print(f"\n共生成 {len(_score_cols)} 个评分列")

In [ ]:
# 逐簇汇总基因集评分——均值 + 阳性细胞百分比。
# 为什么算 per-cluster mean + pct_pos？单个细胞的评分有噪声，
# 簇级汇总能更稳健地反映该簇的整体标记物信号。
# 模式来自 student-code/.../6.3_UCell_cluster_mean_score&pct_pos.py。

if _score_cols and LEIDEN_COL in adata.obs.columns:
    _records = []
    # 兜底：reload 后 leiden 列可能非 categorical
    if not hasattr(adata.obs[LEIDEN_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LEIDEN_COL]):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype("category")
    for _cid in sorted(adata.obs[LEIDEN_COL].cat.categories):
        _mask = adata.obs[LEIDEN_COL] == _cid
        _row = {"cluster": _cid, "n_cells": _mask.sum()}
        for col in _score_cols:
            _vals = adata.obs.loc[_mask, col]
            _ct_name = col.removeprefix("score_")
            _row[f"{_ct_name}_mean"] = round(float(_vals.mean()), 4)
            _row[f"{_ct_name}_pct_pos"] = round(
                float((_vals > 0).mean() * 100), 1
            )
        _records.append(_row)
    _score_summary = pd.DataFrame(_records).set_index("cluster")
    print("基因集评分簇汇总表（前 8 列）:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(_score_summary.iloc[:, :8])
    except ImportError:
        print(_score_summary.iloc[:, :8])
    _score_summary.to_csv("results/figures/stage6_gene_set_scoring_summary.csv")
    print("  汇总表已保存: results/figures/stage6_gene_set_scoring_summary.csv")
    adata.uns["gene_set_scoring_v1"] = {
        "score_columns": _score_cols,
        "method": "scanpy score_genes",
        "marker_csv": MARKER_CSV,
        "timestamp": datetime.datetime.now().isoformat(),
    }
else:
    print("无评分列或缺少 leiden 列，跳过汇总")

## 方法 4：scANVI 标签迁移（守卫——有参考 atlas 时启用）

**前提**：存在对应 `disease_system` 的有标注参考 atlas。
当前胃粘膜没有公认的标注参考，**自动跳过**。
后续若 CELLxGENE Census 或合作者提供有标注的胃参考数据，
填入 `REFERENCE_ATLAS_PATH` 即可启用。

In [ ]:
# scANVI 标签迁移（守卫——REFERENCE_ATLAS_PATH 不存在则优雅跳过）。
# 为什么用 scANVI 而非直接 kNN 映射？scANVI 同时用参考标签和
# 目标数据无监督结构做半监督训练，标签迁移比简单 kNN 更稳健。

if REFERENCE_ATLAS_PATH and os.path.exists(REFERENCE_ATLAS_PATH):
    import scvi

    print(f"加载参考 atlas: {REFERENCE_ATLAS_PATH}")
    _ref = sc.read_h5ad(REFERENCE_ATLAS_PATH)
    print(f"  参考: {_ref.n_obs:,} 细胞 x {_ref.n_vars:,} 基因")

    # 对齐基因集（参考与目标的交集）
    _common = adata.var_names.intersection(_ref.var_names)
    print(f"  共同基因: {len(_common)}")
    _adata_q = adata[:, _common].copy()
    _ref_sub = _ref[:, _common].copy()

    # 设置 scANVI
    scvi.model.SCANVI.setup_anndata(_adata_q, batch_key="source_dataset")
    scvi.model.SCANVI.setup_anndata(_ref_sub, batch_key="source_dataset")
    _model = scvi.model.SCANVI(
        _ref_sub, _adata_q,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )
    _model.train(max_epochs=50, early_stopping=True)

    # 预测 + 概率
    _preds = _model.predict(_adata_q)
    adata.obs["cell_type_scanvi_v1"] = _preds["cell_type"].values
    adata.obs["cell_type_scanvi_v1_uncertainty"] = (
        1.0 - _preds["cell_type"].probabilities.max(axis=1)
    )
    adata.uns["scanvi_v1"] = {
        "reference_atlas": REFERENCE_ATLAS_PATH,
        "method": "scANVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }
    print(f"scANVI 完成: {adata.obs['cell_type_scanvi_v1'].nunique()} 类")
else:
    _reason = (
        "未配置 REFERENCE_ATLAS_PATH" if not REFERENCE_ATLAS_PATH
        else f"{REFERENCE_ATLAS_PATH} 不存在"
    )
    print("=" * 60)
    print(f"scANVI 标签迁移已跳过——原因: {_reason}")
    print("如需启用：设置 REFERENCE_ATLAS_PATH 为有标注的 .h5ad 路径，")
    print("并确保参考数据包含 cell_type obs 列。")
    print("=" * 60)
    # 预建空列保持结构完整
    adata.obs["cell_type_scanvi_v1"] = np.nan
    adata.obs["cell_type_scanvi_v1"] = (
        adata.obs["cell_type_scanvi_v1"].astype("category")
    )

## 方法 5（候选，已注释）：CellTypist 预训练分类器

CellTypist 提供多个预训练模型（如 `Immune_All_Low.pkl`、`Developing_Mouse_Brain.pkl` 等）。
**当前注释原因**：PI 的胃/滑膜系统目前没有很好匹配的预训练模型。
当有匹配模型出现时取消注释即可启用。新列自动进入跨方法比较。

In [ ]:
# === CellTypist（候选——有对应预训练模型时取消注释启用）===
# 前提：pip install celltypist
# 当存在对应组织的 CellTypist 预训练模型时取消注释：
# # import celltypist
# # predictions = celltypist.annotate(
# #     adata, model="Human_Gastric_Atlas.pkl",
# #     majority_voting=True,
# # )
# # adata.obs["cell_type_celltypist_v1"] = (
# #     predictions.predicted_labels["majority_voting"].values
# # )
# # print(f"CellTypist: {adata.obs['cell_type_celltypist_v1'].nunique()} 类")
print(
    "CellTypist cell is commented out. "
    "Uncomment when a suitable pretrained model is available for this tissue."
)

## 跨方法比较：混淆矩阵 + Cohen's kappa + Sankey

对已产生标签的任意两种方法，计算混淆矩阵和 Cohen's kappa，
并用 Sankey 图可视化标签流。
**为什么做跨方法比较？** 一致的方法增强置信度；分歧的方法揭示
需要 PI 重点复核的簇——这才是多方法并跑的核心价值。

In [ ]:
# 收集所有已产生的标注列（不含纯数字、不含空列）。
_label_cols = []
for col in sorted(adata.obs.columns):
    if not any(kw in col for kw in ("cell_type", "_v1")):
        continue
    if col.endswith("_proportion") or col.endswith("_entropy"):
        continue
    if col == "cell_type_final_v1":
        continue  # 最终标签由 PI 拍板，不参与双向比较
    # 排除全 NaN 列
    if adata.obs[col].notna().sum() == 0:
        continue
    # 排除数值列（如 uncertainty）
    if adata.obs[col].dtype == "float64" and "uncertainty" in col:
        continue
    _label_cols.append(col)

print(f"可用标注列 ({len(_label_cols)}):")
for col in _label_cols:
    _n = adata.obs[col].nunique()
    print(f"  {col}  ({_n} 类)")
    # 确保是 category 类型
    if adata.obs[col].dtype.name != "category":
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

In [ ]:
# 成对混淆矩阵 + Cohen's kappa。
# 从 scorers 直接调用 annotation_concordance——透明、无回调。
from itertools import combinations

if len(_label_cols) >= 2:
    _kappa_results = []
    for _a, _b in combinations(_label_cols, 2):
        # 只比较都有有效标签的细胞
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 2:
            continue
        _k = annotation_concordance(adata, label_a=_a, label_b=_b)
        _kappa = _k.get("cohen_kappa", np.nan)
        _kappa_results.append({
            "method_a": _a, "method_b": _b,
            "cohen_kappa": round(float(_kappa), 4),
            "n_cells": int(_valid.sum()),
        })
        print(f"  {_a} vs {_b}: kappa={_kappa:.4f} (n={_valid.sum():,})")

    _kappa_df = pd.DataFrame(_kappa_results)
    if len(_kappa_df) > 0:
        print(f"\nCohen's kappa 汇总:")
        try:
            from IPython.display import display as ipy_display
            ipy_display(_kappa_df)
        except ImportError:
            print(_kappa_df)
        # 成对 kappa 写 CSV 为规范记录（h5ad uns 序列化复杂嵌套对象不稳定）
        _kappa_df.to_csv("results/figures/stage6_kappa_pairs.csv", index=False)
        print("  成对 kappa 表已保存: results/figures/stage6_kappa_pairs.csv")
        adata.uns["cross_method_comparison_v1"] = {
            "label_columns": _label_cols,
            "kappa_csv": "results/figures/stage6_kappa_pairs.csv",
            "timestamp": datetime.datetime.now().isoformat(),
        }
else:
    print("可用标注列 <2，跳过成对比较")

In [ ]:
# Sankey 图——可视化两种方法之间的标签流。
# 每个节点=一种细胞类型标签，连线宽度=细胞数。
# 为什么用 Sankey？直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"。

if len(_label_cols) >= 2:
    for _a, _b in combinations(_label_cols, 2):
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 10:
            continue
        # 构建流向表（method_a -> method_b）
        _flow = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
        )
        # 保存流向矩阵
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/stage6_sankey/flow_{_safe_a}__vs__{_safe_b}.csv"
        _flow.to_csv(_fname)
        print(f"  流向矩阵已保存: {_fname} (shape={_flow.shape})")

    print(f"\nSankey 流向矩阵已保存至 results/figures/stage6_sankey/")
    print("（用 R/plotly 等工具绘制 Sankey；Python matplotlib.sankey 不稳健，略过绘图）")
else:
    print("可用标注列 <2，跳过 Sankey")

## LLM 逐簇综合判决（key 守卫——无 key 跳过）

构造包含以下信息的中文 prompt，调用 LLM 逐簇给出综合判决：
- 各方法（marker / LLM 共识 / 基因集评分 / scANVI / CellTypist）对该簇的标签
- 原始作者标注（如 `cell_type_original_*` 存在）
- Top 标记基因（`sc.tl.rank_genes_groups`）
- 基因集评分 profile
- 各 source_dataset 的 qc_skipped 记录

每簇判决写为独立 markdown 文件：`results/figures/stage6_verdicts/cluster_{id}.md`

**状态：代码已写，待 PI 在 `.env` 配 key 后人工运行调试。**
无 key 时优雅跳过，notebook 不崩溃。

In [ ]:
# LLM 逐簇综合判决（key 守卫——无 key 跳过）。
# 每簇独立调用 LLM，避免上下文过长导致输出质量下降。
import json

# 检查 LLM key 是否可用
from dotenv import load_dotenv
load_dotenv()

_llm_providers = ["OPENAI", "ANTHROPIC", "DEEPSEEK", "QWEN", "GEMINI"]
_has_key = any(
    os.getenv(f"{p}_API_KEY") not in (None, "")
    for p in _llm_providers
)

if _has_key:
    # 确保有 rank_genes_groups 结果（可能方法 2 没跑）
    if "rank_genes_stage6" not in adata.uns:
        sc.tl.rank_genes_groups(
            adata, groupby=LEIDEN_COL, method="wilcoxon",
            n_genes=30, key_added="rank_genes_stage6",
        )

    # 按 VERDICT_MODEL 前缀选 provider 创建客户端
    _pfx = VERDICT_MODEL.split("/")[0].lower()
    if _pfx in ("openai", "qwen", "deepseek"):
        from openai import OpenAI
        _key_var = f"{_pfx.upper()}_API_KEY"
        _base_var = f"{_pfx.upper()}_BASE_URL"
        _key = os.getenv(_key_var)
        _base = os.getenv(_base_var) or None
        if not _key:
            print(f"{_pfx} API key 未配置，跳过 LLM verdict")
        else:
            _client = OpenAI(api_key=_key, base_url=_base)

            # 收集已有标注列信息
            _annotation_info = []
            for col in adata.obs.columns:
                if not any(kw in col for kw in ("cell_type", "_v1")):
                    continue
                if col.endswith("_proportion") or col.endswith("_entropy"):
                    continue
                if col == "cell_type_final_v1":
                    continue
                _present = adata.obs[col].notna().sum()
                if _present > 0:
                    _annotation_info.append(col)

            # 收集基因集评分列
            _score_info = [c for c in adata.obs.columns if c.startswith("score_")]

            # qc_skipped 上下文（从 adata.uns 取，如不存在则跳过）
            _qc_ctx = adata.uns.get("qc_skipped", {})
            _qc_str = json.dumps(_qc_ctx, ensure_ascii=False, indent=2)

            # 逐簇调用 LLM
            _cluster_ids = sorted(adata.obs[LEIDEN_COL].cat.categories)
            print(f"逐簇 LLM 判决 ({len(_cluster_ids)} 个簇)...")

            for _cid in _cluster_ids:
                # 构建该簇的上下文
                _df = sc.get.rank_genes_groups_df(
                    adata, group=_cid, key="rank_genes_stage6"
                )
                _top_genes = _df["names"].head(15).tolist()
                _top_scores = _df["scores"].head(15).tolist()

                # 各方法对该簇的标签
                _mask = adata.obs[LEIDEN_COL] == _cid
                _method_labels = {}
                for col in _annotation_info:
                    _val = adata.obs.loc[_mask, col].mode()
                    if len(_val) > 0 and pd.notna(_val.iloc[0]):
                        _method_labels[col] = str(_val.iloc[0])

                # 基因集评分 profile
                _score_profile = {}
                for col in _score_info:
                    _score_profile[col] = round(
                        float(adata.obs.loc[_mask, col].mean()), 4
                    )

                # 构造中文 prompt
                _gene_lines = "\n".join(
                    f"- {g}: logFC={s:.2f}" for g, s in zip(_top_genes, _top_scores)
                )
                _prompt = (
                    f"你是单细胞转录组学专家。请判断 Leiden 簇 {_cid} 的细胞类型。\n\n"
                    f"## 上下文\n"
                    f"- 物种: human\n"
                    f"- 组织: stomach（胃粘膜）\n"
                    f"- 簇大小: {_mask.sum()} 细胞\n\n"
                    f"## 各方法标签\n"
                    f"{json.dumps(_method_labels, ensure_ascii=False, indent=2)}\n\n"
                    f"## Top 标记基因（按 fold-change 排序）\n"
                    f"{_gene_lines}\n\n"
                    f"## 基因集评分（簇均值）\n"
                    f"{json.dumps(_score_profile, ensure_ascii=False, indent=2)}\n\n"
                    f"## 上游数据集 QC 记录\n"
                    f"{_qc_str}\n\n"
                    f"## 任务\n"
                    f"1. 给出最可能的细胞类型（具体到亚型，如 CD4+ Tcm 而非 T_cell）\n"
                    f"2. 置信度（high/medium/low）及理由\n"
                    f"3. 各方法分歧的原因分析\n"
                    f"4. 建议 PI 重点复核什么\n\n"
                    f"用中文回答，结构清晰。"
                )

                try:
                    _resp = _client.chat.completions.create(
                        model=VERDICT_MODEL,
                        messages=[
                            {"role": "system",
                             "content": "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱注释。"},
                            {"role": "user", "content": _prompt},
                        ],
                        temperature=0.3,
                        max_tokens=1500,
                    )
                    _verdict = _resp.choices[0].message.content
                except Exception as _e:
                    _verdict = f"LLM 调用失败: {_e}"

                # 写 verdict markdown
                _out_path = f"results/figures/stage6_verdicts/cluster_{_cid}.md"
                with open(_out_path, "w") as _f:
                    _f.write(f"# 簇 {_cid} LLM 综合判决\n\n")
                    _f.write(f"**模型**: {VERDICT_MODEL}\n\n")
                    _f.write(f"**簇大小**: {_mask.sum()} 细胞\n\n")
                    _f.write(f"**各方法标签**:\n")
                    for _m, _lbl in _method_labels.items():
                        _f.write(f"- {_m}: {_lbl}\n")
                    _f.write(f"\n---\n\n{_verdict}\n")
                print(f"  簇 {_cid}: verdict -> {_out_path}")

            adata.uns["llm_verdict_v1"] = {
                "model": VERDICT_MODEL,
                "output_dir": "results/figures/stage6_verdicts",
                "n_clusters": len(_cluster_ids),
                "timestamp": datetime.datetime.now().isoformat(),
            }
    elif _pfx == "anthropic":
        from anthropic import Anthropic
        _key = os.getenv("ANTHROPIC_API_KEY")
        _base = os.getenv("ANTHROPIC_BASE_URL") or None
        if not _key:
            print("Anthropic API key 未配置，跳过 LLM verdict")
        else:
            print(
                "Anthropic verdict 模式（代码结构同上）——"
                "待 PI 实现或改用 OpenAI 兼容模式"
            )
    else:
        print(
            f"VERDICT_MODEL 的 provider '{_pfx}' 暂未支持，",
            "请在 PARAMS 选 openai/anthropic/deepseek/qwen"
        )
else:
    print("=" * 60)
    print("LLM 逐簇判决已跳过——未检测到 LLM API key。")
    print("请在项目根目录 .env 文件中配置至少一家 provider 的 API key，")
    print("然后重新运行本 cell。模板见 .env.example。")
    print("=" * 60)

## PI 拍板：`cell_type_final_v1`

PI 阅读每簇的 LLM 判决 markdown 后，手动决定最终标签。
**为什么不让 LLM 自动拍板？** 每个簇的最终标签是科学判断，
LLM 是顾问不是决策者——PI 结合自身领域知识复核后决定。
对胃癌前病变项目（~20-30 簇），阅读所有判决约需 30-60 分钟。

In [ ]:
# === PI 拍板区 ===
# PI 阅读 results/figures/stage6_verdicts/cluster_{id}.md 后，
# 在下方的 pi_decisions 字典中填入每个簇的最终细胞类型。
# 示例（基于 Nowicki 数据——PI 需修改为实际判断）：
pi_decisions = {
    # "0": "B_cell",
    # "1": "CD4_T_cell",
    # "2": "pit_cell",
    # ...  PI 逐簇填入
}

if pi_decisions:
    adata.obs["cell_type_final_v1"] = (
        adata.obs[LEIDEN_COL].astype(str).map(pi_decisions)
    )
    n_final = adata.obs["cell_type_final_v1"].notna().sum()
    print(f"最终标注: {n_final}/{adata.n_obs} 细胞已标注")
    print(f"  标签种类: {adata.obs['cell_type_final_v1'].nunique()}")
else:
    # PI 暂未填写，预建空列
    adata.obs["cell_type_final_v1"] = np.nan
    adata.obs["cell_type_final_v1"] = (
        adata.obs["cell_type_final_v1"].astype("category")
    )
    print("PI 暂未拍板，已预建 cell_type_final_v1 空列")

In [ ]:
# 记录 PI 拍板的元数据——plain adata.uns 写入。
# 为什么用 plain dict？框架不预设结构，PI 自由记录。
adata.uns["cell_type_final_v1_notes"] = {
    "leiden_resolution_used": LEIDEN_COL,
    "available_methods": _label_cols if "_label_cols" in dir() else [],
    "method_basis": "PI manual review of LLM verdicts + marker dotplot",
    "rationale": "PI reviewed each cluster verdict; final labels reflect domain expertise",
    "timestamp": datetime.datetime.now().isoformat(),
}
print("PI 拍板元数据已写入 adata.uns['cell_type_final_v1_notes']")

In [ ]:
# 统一追踪字段——stage + version + upstream（与 stage4-5-7 命名一致）
adata.uns["stage"] = "stage6_annotated"    # 本 stage 标识
adata.uns["version"] = "v1"                 # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]     # list 形式，支持多上游合并
adata.uns["status"] = "experimental"        # PI changes to "promoted" after review

# 记录 stage 6 运行元数据（嵌套 dict——细节层，与顶层追踪字段并存）。
adata.uns["stage6_v1"] = {
    "upstream": UPSTREAM_PATH,
    "leiden_col": LEIDEN_COL,
    "marker_csv": MARKER_CSV,
    "methods_run": [
        "marker_dotplot",
        "mllmcelltype_consensus",
        "gene_set_scoring",
    ],
    "scANVI_skipped": not bool(REFERENCE_ATLAS_PATH),
    "timestamp": datetime.datetime.now().isoformat(),
}
print("Stage 6 运行元数据已记录")

In [ ]:
# 内存纪律自检——写入前一次断言，守卫最高影响的内存退化。
# 只守卫 adata.X（cells x genes，尺寸最大）；obsm 矩阵
# 小一个量级，回归由 PR review 捕获。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

In [ ]:
# 写出 stage checkpoint（内存纪律：lzf 压缩）。
# 为什么 lzf？比 gzip 快，比不压缩小约 30%，且保留 sparse CSR。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")

assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存跨越 stage 边界（内存纪律：del + gc.collect）。
# 如果不释放，Jupyter kernel 会一直持有上一 stage 的 adata，
# 后续 stage 累积 OOM。
del adata
gc.collect()
print("内存已释放")